# Q4

We implement the **multidimensional Newton iteration** for a system
$$
F:\mathbb{R}^N \to \mathbb{R}^N,
$$
and then apply it to the system
$$
x^2-y=1,\qquad x-y^2=-1.
$$

The Newton step for a current iterate $x_k\in\mathbb{R}^N$ is obtained by solving
$$
J(x_k)\,s_k = -F(x_k),
$$
where $J(x_k)$ is the Jacobian matrix of $F$.  
We then update
$$
x_{k+1}=x_k+s_k.
$$

As in the attached notebook, the code prints the iterates so that the convergence can be seen directly.

## The Newton iteration we will use

The function below is called as

```python
newton(F, J, x0, tol)
```

where:

- `F` returns the residual vector,
- `J` returns the Jacobian matrix,
- `x0` is the initial guess,
- `tol` is the stopping tolerance.

We stop when
$$
\|F(x_k)\|_\infty < \texttt{tol}.
$$

The linear system is solved with `numpy.linalg.solve`.

In [1]:
import numpy as np
import sympy as sp

def newton(F, J, x0, tol, maxit=25):
    x = np.array(x0, dtype=float)
    Fx = np.array(F(x), dtype=float)
    iterations = 0
    print(f"Iteration {iterations:2d}: x = {x}   ||F(x)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

    while np.linalg.norm(Fx, ord=np.inf) > tol and iterations < maxit:
        iterations += 1
        s = np.linalg.solve(np.array(J(x), dtype=float), -Fx)
        x = x + s
        Fx = np.array(F(x), dtype=float)
        print(f"Iteration {iterations:2d}: x = {x}   ||F(x)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

    return x

## Defining the system and its Jacobian

Write the system in residual form:
$$
F(x,y)=
\begin{pmatrix}
x^2-y-1\\
x-y^2+1
\end{pmatrix}.
$$

Its Jacobian is
$$
J(x,y)=
\begin{pmatrix}
2x & -1\\
1 & -2y
\end{pmatrix}.
$$

Below, `sympy` is used to generate the Jacobian matrix symbolically, and then we convert both the residual and the Jacobian into callable Python functions.

In [2]:
# Symbolic construction with sympy
x_sym, y_sym = sp.symbols('x y')
F_sym = sp.Matrix([
    x_sym**2 - y_sym - 1,
    x_sym - y_sym**2 + 1
])
J_sym = F_sym.jacobian([x_sym, y_sym])

display(F_sym)
display(J_sym)

# Convert to numerical callables
F_lam = sp.lambdify((x_sym, y_sym), F_sym, 'numpy')
J_lam = sp.lambdify((x_sym, y_sym), J_sym, 'numpy')

def F(v):
    x, y = v
    return np.array(F_lam(x, y), dtype=float).reshape(2)

def J(v):
    x, y = v
    return np.array(J_lam(x, y), dtype=float)

Matrix([
[x**2 - y - 1],
[x - y**2 + 1]])

Matrix([
[2*x,   -1],
[  1, -2*y]])

## Part (b): finding solutions with different initial guesses

We now apply Newton's method from several starting points.

Because Newton's method for nonlinear systems can converge to different roots depending on the initial guess, trying more than one $x_0$ is the natural way to locate multiple solutions.

In [3]:
print("Starting from x0 = (0.2, -0.8)")
print("-" * 80)
root1 = newton(F, J, [0.2, -0.8], 1e-12)
print(f"Approximate root = {root1}")

print("\nStarting from x0 = (-0.8, 0.2)")
print("-" * 80)
root2 = newton(F, J, [-0.8, 0.2], 1e-12)
print(f"Approximate root = {root2}")

Starting from x0 = (0.2, -0.8)
--------------------------------------------------------------------------------
Iteration  0: x = [ 0.2 -0.8]   ||F(x)||_inf = 5.600000e-01
Iteration  1: x = [ 0.01463415 -1.03414634]   ||F(x)||_inf = 5.482451e-02
Iteration  2: x = [-6.81760099e-04 -1.00023411e+00]   ||F(x)||_inf = 1.150039e-03
Iteration  3: x = [ 8.77396030e-07 -1.00000047e+00]   ||F(x)||_inf = 4.659939e-07
Iteration  4: x = [ 1.32240973e-12 -1.00000000e+00]   ||F(x)||_inf = 7.698286e-13
Approximate root = [ 1.32240973e-12 -1.00000000e+00]

Starting from x0 = (-0.8, 0.2)
--------------------------------------------------------------------------------
Iteration  0: x = [-0.8  0.2]   ||F(x)||_inf = 5.600000e-01
Iteration  1: x = [-1.03414634  0.01463415]   ||F(x)||_inf = 5.482451e-02
Iteration  2: x = [-1.00023411e+00 -6.81760099e-04]   ||F(x)||_inf = 1.150039e-03
Iteration  3: x = [-1.00000047e+00  8.77396030e-07]   ||F(x)||_inf = 4.659939e-07
Iteration  4: x = [-1.00000000e+00  1.322404

These two runs already produce two distinct solutions:

- approximately $(0,-1)$,
- approximately $(-1,0)$.

To see how many solutions there are in total, we can try other initial guesses as well.

In [4]:
print("Starting from x0 = (2, 2)")
print("-" * 80)
root3 = newton(F, J, [2, 2], 1e-12)
print(f"Approximate root = {root3}")

print("\nStarting from x0 = (-1, -1)")
print("-" * 80)
root4 = newton(F, J, [-1, -1], 1e-12)
print(f"Approximate root = {root4}")

Starting from x0 = (2, 2)
--------------------------------------------------------------------------------
Iteration  0: x = [2. 2.]   ||F(x)||_inf = 1.000000e+00
Iteration  1: x = [1.66666667 1.66666667]   ||F(x)||_inf = 1.111111e-01
Iteration  2: x = [1.61904762 1.61904762]   ||F(x)||_inf = 2.267574e-03
Iteration  3: x = [1.61803445 1.61803445]   ||F(x)||_inf = 1.026516e-06
Iteration  4: x = [1.61803399 1.61803399]   ||F(x)||_inf = 2.104983e-13
Approximate root = [1.61803399 1.61803399]

Starting from x0 = (-1, -1)
--------------------------------------------------------------------------------
Iteration  0: x = [-1. -1.]   ||F(x)||_inf = 1.000000e+00
Iteration  1: x = [-0.66666667 -0.66666667]   ||F(x)||_inf = 1.111111e-01
Iteration  2: x = [-0.61904762 -0.61904762]   ||F(x)||_inf = 2.267574e-03
Iteration  3: x = [-0.61803445 -0.61803445]   ||F(x)||_inf = 1.026516e-06
Iteration  4: x = [-0.61803399 -0.61803399]   ||F(x)||_inf = 2.107203e-13
Approximate root = [-0.61803399 -0.6180339

The additional runs suggest two more solutions, both lying on the line $x=y$.

To confirm the total number of solutions, we can also solve the system algebraically.

## Counting all solutions exactly

From the first equation,
$$
y=x^2-1.
$$
Substitute this into the second equation:
$$
x-(x^2-1)^2=-1.
$$
Rearranging gives
$$
x^4-2x^2-x=0
$$
and hence
$$
x(x^3-2x-1)=0.
$$

The cubic factor splits as
$$
x^3-2x-1=(x+1)(x^2-x-1).
$$
Therefore
$$
x\in\left\{0,\,-1,\,\frac{1+\sqrt5}{2},\,\frac{1-\sqrt5}{2}\right\}.
$$

Using $y=x^2-1$, each corresponding $y$ is the same value as $x$ for the last two roots, and we obtain all solutions.

In [5]:
phi_plus = (1 + np.sqrt(5)) / 2
phi_minus = (1 - np.sqrt(5)) / 2

exact_solutions = [
    np.array([0.0, -1.0]),
    np.array([-1.0, 0.0]),
    np.array([phi_plus, phi_plus]),
    np.array([phi_minus, phi_minus]),
]

print("All solutions:")
for sol in exact_solutions:
    print(sol)

All solutions:
[ 0. -1.]
[-1.  0.]
[1.61803399 1.61803399]
[-0.61803399 -0.61803399]


## Summary

Using the multidimensional Newton method, we can find at least two solutions immediately by choosing different initial guesses, for example:

- from $(0.2,-0.8)$, Newton converges to $(0,-1)$;
- from $(-0.8,0.2)$, Newton converges to $(-1,0)$.

Trying further initial guesses reveals the other two roots:
$$
\left(\frac{1+\sqrt5}{2},\frac{1+\sqrt5}{2}\right),
\qquad
\left(\frac{1-\sqrt5}{2},\frac{1-\sqrt5}{2}\right).
$$

So the system has **four real solutions** in total:
$$
(0,-1),\quad (-1,0),\quad
\left(\frac{1+\sqrt5}{2},\frac{1+\sqrt5}{2}\right),\quad
\left(\frac{1-\sqrt5}{2},\frac{1-\sqrt5}{2}\right).
$$

## Q4(ii): minimising the Rosenbrock energy

We now consider the energy
$$
E:\mathbb{R}^2\to\mathbb{R},\qquad
E(x,y)=(1-x)^2+100\,(y-x^2)^2.
$$

### (a) Global minimum by inspection

Both terms are squares, so
$$
E(x,y)\ge 0
$$
for all $(x,y)$.

The minimum possible value is therefore $0$, and this occurs exactly when both squares vanish:
$$
1-x=0,\qquad y-x^2=0.
$$
Hence
$$
x=1,\qquad y=1.
$$

So the **global minimiser** is
$$
(x,y)=(1,1),
$$
and the minimum energy is
$$
E(1,1)=0.
$$


In [6]:
# Symbolic gradient and Hessian of the Rosenbrock energy
xr, yr = sp.symbols('xr yr')
E_sym = (1 - xr)**2 + 100*(yr - xr**2)**2

gradE_sym = sp.Matrix([sp.diff(E_sym, xr), sp.diff(E_sym, yr)])
HE_sym = gradE_sym.jacobian([xr, yr])

display(E_sym)
display(gradE_sym)
display(HE_sym)

gradE_lam = sp.lambdify((xr, yr), gradE_sym, 'numpy')
HE_lam = sp.lambdify((xr, yr), HE_sym, 'numpy')

def gradE(v):
    x, y = v
    return np.array(gradE_lam(x, y), dtype=float).reshape(2)

def HE(v):
    x, y = v
    return np.array(HE_lam(x, y), dtype=float)

def grad_norm(v):
    g = np.array(gradE(v), dtype=float)
    return np.sqrt(np.dot(g, g))


(1 - xr)**2 + 100*(-xr**2 + yr)**2

Matrix([
[-400*xr*(-xr**2 + yr) + 2*xr - 2],
[             -200*xr**2 + 200*yr]])

Matrix([
[1200*xr**2 - 400*yr + 2, -400*xr],
[                -400*xr,     200]])

### (b) Newton's method applied to the equation $\nabla E(x,y)=0$

To locate stationary points, we solve
$$
\nabla E(x,y)=0.
$$
This is a nonlinear system in two variables, so Newton's method uses the Hessian of $E$ as the Jacobian of $\nabla E$:
$$
H_E(x_k)\,s_k = -\nabla E(x_k),\qquad
x_{k+1}=x_k+s_k.
$$

The stopping criterion requested in the question is
$$
\sqrt{\nabla E(x_k)\cdot \nabla E(x_k)} < 10^{-8}.
$$


In [7]:
def newton_on_gradient(x0, tol=1e-8, maxit=100, verbose=True):
    x = np.array(x0, dtype=float)
    g = gradE(x)
    it = 0

    if verbose:
        print(f"Iteration {it:2d}: x = {x}   ||grad E||_2 = {np.linalg.norm(g):.6e}")

    while np.linalg.norm(g) > tol and it < maxit:
        s = np.linalg.solve(HE(x), -g)
        x = x + s
        g = gradE(x)
        it += 1
        if verbose:
            print(f"Iteration {it:2d}: x = {x}   ||grad E||_2 = {np.linalg.norm(g):.6e}")

    return x, it, np.linalg.norm(g)

newton_starts = [(2, 2), (10, 10), (100, 100)]
newton_results = {}

for start in newton_starts:
    print(f"Starting from x0 = {start}")
    print("-" * 80)
    x_star, iters, gnorm = newton_on_gradient(start, tol=1e-8, maxit=100, verbose=True)
    newton_results[start] = (x_star, iters, gnorm)
    print(f"Converged to {x_star} in {iters} iterations with ||grad E||_2 = {gnorm:.6e}\n")


Starting from x0 = (2, 2)
--------------------------------------------------------------------------------
Iteration  0: x = [2. 2.]   ||grad E||_2 = 1.651183e+03
Iteration  1: x = [1.99750623 3.99002494]   ||grad E||_2 = 1.999982e+00
Iteration  2: x = [1.00123913 0.00993165]   ||grad E||_2 = 4.443233e+02
Iteration  3: x = [1.00123292 1.00246736]   ||grad E||_2 = 2.465855e-03
Iteration  4: x = [1.         0.99999848]   ||grad E||_2 = 6.798053e-04
Iteration  5: x = [1. 1.]   ||grad E||_2 = 5.773160e-15
Converged to [1. 1.] in 5 iterations with ||grad E||_2 = 5.773160e-15

Starting from x0 = (10, 10)
--------------------------------------------------------------------------------
Iteration  0: x = [10. 10.]   ||grad E||_2 = 3.604677e+05
Iteration  1: x = [ 9.99950003 99.99000056]   ||grad E||_2 = 1.800000e+01
Iteration  2: x = [  1.0004499  -79.98200315]   ||grad E||_2 = 3.622969e+04
Iteration  3: x = [1.00044987 1.00089995]   ||grad E||_2 = 8.997494e-04
Iteration  4: x = [1.        0.99

For these three initial guesses, Newton's method converges to the minimiser $(1,1)$ in **5 iterations** in each case.

This is typical of Newton's method near a nondegenerate critical point: once it gets close, the convergence is very fast.


### (c) Gradient descent from the approximation $J(x)=\alpha I$

If we replace the Jacobian in Newton's method by the approximation
$$
J(x)=\alpha I,\qquad \alpha>0,
$$
then the linear system becomes
$$
\alpha I\,s_k = -\nabla E(x_k).
$$
So
$$
s_k = -\frac{1}{\alpha}\nabla E(x_k),
$$
and the update is
$$
x_{k+1}=x_k-\frac{1}{\alpha}\nabla E(x_k).
$$

This is gradient descent with learning rate $1/\alpha$.

We again stop when
$$
\|\nabla E(x_k)\|_2<10^{-8}.
$$


In [8]:
def gradient_descent_alphaI(x0, alpha, tol=1e-8, maxit=200000, verbose=False):
    x = np.array(x0, dtype=float)
    g = gradE(x)
    it = 0

    if verbose:
        print(f"Iteration {it:6d}: x = {x}   ||grad E||_2 = {np.linalg.norm(g):.6e}")

    while np.linalg.norm(g) > tol and it < maxit:
        x = x - g / alpha
        g = gradE(x)
        it += 1

        if verbose and (it <= 10 or it % 5000 == 0):
            print(f"Iteration {it:6d}: x = {x}   ||grad E||_2 = {np.linalg.norm(g):.6e}")

        if not np.all(np.isfinite(x)) or not np.all(np.isfinite(g)):
            return x, it, np.linalg.norm(g), False

    return x, it, np.linalg.norm(g), (it < maxit)

gd_alphas = [1000, 500, 2000]
gd_results = {}

for alpha in gd_alphas:
    x_star, iters, gnorm, converged = gradient_descent_alphaI((2, 2), alpha, tol=1e-8, maxit=200000, verbose=False)
    gd_results[alpha] = (x_star, iters, gnorm, converged)

    if converged:
        print(f"alpha = {alpha:4d}: converged to {x_star} in {iters} iterations with ||grad E||_2 = {gnorm:.6e}")
    else:
        print(f"alpha = {alpha:4d}: did not converge (iteration count reached {iters}, ||grad E||_2 = {gnorm:.6e})")


alpha = 1000: converged to [1.00000001 1.00000002] in 43802 iterations with ||grad E||_2 = 9.997729e-09
alpha =  500: did not converge (iteration count reached 8, ||grad E||_2 = inf)


<lambdifygenerated-3>:2: RuntimeWarning: overflow encountered in scalar power
  return array([[-400*xr*(-xr**2 + yr) + 2*xr - 2], [-200*xr**2 + 200*yr]])


alpha = 2000: converged to [1.00000001 1.00000002] in 91661 iterations with ||grad E||_2 = 9.998201e-09


From the numerical runs:

- For $\alpha=1000$, gradient descent converges to $(1,1)$, but it takes **43802 iterations**.
- For $\alpha=500$, the method is unstable and diverges very quickly (the iterates blow up numerically).
- For $\alpha=2000$, the method still converges, but the step size is only half as large, so it is slower: **91661 iterations**.

So, compared with Newton's method, the $\alpha I$ approximation is much more robust to implement but dramatically slower here.
